# Historical U-Net experiment, not a clinical model
Not executed end-to-end. Original classification images do not establish segmentation masks. Supply genuine paired annotations. The package CLI is the supported offline path. No automatic downloads, credential handling, or fabricated empty masks.


In [ ]:
# Historical download/setup removed. Supply licensed local images and genuine binary masks.


In [ ]:
# Historical download/setup removed. Supply licensed local images and genuine binary masks.


In [ ]:
# Historical download/setup removed. Supply licensed local images and genuine binary masks.


In [ ]:
# Historical download/setup removed. Supply licensed local images and genuine binary masks.


In [ ]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf

# Define image properties
IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128  # Adjust as needed
IMAGE_CHANNELS = 1  # Grayscale images

# Paths to images and masks
IMAGE_DIR = '/content/brain_tumor_dataset/yes'  # Adjust based on dataset structure
MASK_DIR = '/content/yes'  # Adjust based on dataset structure

def load_data(image_dir, mask_dir, img_size=(IMAGE_HEIGHT, IMAGE_WIDTH)):
    images = []
    masks = []
    image_files = sorted(os.listdir(image_dir))
    mask_files = set(os.listdir(mask_dir))
    missing = set(image_files) - mask_files
    if missing:
        raise ValueError(f'Missing genuine masks for {len(missing)} images; do not invent labels.')

    for img_name in image_files:
        # Load and preprocess the image
        img_path = os.path.join(image_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f'Unreadable image: {img_name}')
        img = cv2.resize(img, img_size)
        img = img / 255.0  # Normalize
        images.append(img[..., np.newaxis])

        # Load and preprocess the mask
        mask_path = os.path.join(mask_dir, img_name)
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise ValueError(f'Unreadable mask: {img_name}')
            mask = cv2.resize(mask, img_size, interpolation=cv2.INTER_NEAREST)
            mask = mask / 255.0  # Normalize
            mask = (mask > 0.5).astype(np.float32)  # Binarize
            masks.append(mask[..., np.newaxis])
        else:
            raise ValueError(f'Missing genuine mask: {img_name}')

    return np.array(images), np.array(masks)

# Load images and masks
X, y = load_data(IMAGE_DIR, MASK_DIR)

print(f'Images shape: {X.shape}')
print(f'Masks shape: {y.shape}')


In [ ]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape}, {y_train.shape}')
print(f'Validation set: {X_val.shape}, {y_val.shape}')


In [ ]:
# Historical augmentation removed: image/mask pairing and categorical mask
# interpolation were not verified. Use the explicit package CLI for local training.
train_gen = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(16)
steps_per_epoch = None


In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def unet_model(input_size=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)):
    inputs = Input(input_size)

    # Downsampling
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(conv4)
    drop4 = Dropout(0.5)(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(drop4)

    # Bottleneck
    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(pool4)
    conv5 = Conv2D(1024, 3, activation='relu', padding='same')(conv5)
    drop5 = Dropout(0.5)(conv5)

    # Upsampling
    up6 = UpSampling2D(size=(2, 2))(drop5)
    up6 = Conv2D(512, 2, activation='relu', padding='same')(up6)
    merge6 = concatenate([drop4, up6], axis=3)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(merge6)
    conv6 = Conv2D(512, 3, activation='relu', padding='same')(conv6)

    up7 = UpSampling2D(size=(2, 2))(conv6)
    up7 = Conv2D(256, 2, activation='relu', padding='same')(up7)
    merge7 = concatenate([conv3, up7], axis=3)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(merge7)
    conv7 = Conv2D(256, 3, activation='relu', padding='same')(conv7)

    up8 = UpSampling2D(size=(2, 2))(conv7)
    up8 = Conv2D(128, 2, activation='relu', padding='same')(up8)
    merge8 = concatenate([conv2, up8], axis=3)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(merge8)
    conv8 = Conv2D(128, 3, activation='relu', padding='same')(conv8)

    up9 = UpSampling2D(size=(2, 2))(conv8)
    up9 = Conv2D(64, 2, activation='relu', padding='same')(up9)
    merge9 = concatenate([conv1, up9], axis=3)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(merge9)
    conv9 = Conv2D(64, 3, activation='relu', padding='same')(conv9)
    conv9 = Conv2D(2, 3, activation='relu', padding='same')(conv9)

    conv10 = Conv2D(1, 1, activation='sigmoid')(conv9)

    model = Model(inputs=inputs, outputs=conv10)

    return model

# Instantiate the model
model = unet_model()

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

# Display the model architecture
model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Define callbacks
checkpoint = ModelCheckpoint(
    'unet_brain_tumor.h5',
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)
earlystop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.1,
    patience=5,
    verbose=1,
    min_lr=1e-6
)

callbacks_list = [checkpoint, earlystop, reduce_lr]

# Train the model
epochs = 20

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list
)


In [ ]:
import matplotlib.pyplot as plt

# Load the best model
model.load_weights('unet_brain_tumor.h5')

# Evaluate on validation data
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Loss: {loss}')
print(f'Validation Accuracy: {accuracy}')

# Plot training & validation loss values
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot training & validation accuracy values
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()


In [ ]:
def predict_and_display(model, image, mask=None):
    # Predict the mask
    pred_mask = model.predict(image[np.newaxis, ...])[0]
    pred_mask = (pred_mask > 0.5).astype(np.uint8)

    plt.figure(figsize=(12, 6))

    # Original Image
    plt.subplot(1, 3, 1)
    plt.imshow(image.squeeze(), cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    # Ground Truth Mask
    if mask is not None:
        plt.subplot(1, 3, 2)
        plt.imshow(mask.squeeze(), cmap='gray')
        plt.title('Ground Truth Mask')
        plt.axis('off')

    # Predicted Mask
    plt.subplot(1, 3, 3)
    plt.imshow(pred_mask.squeeze(), cmap='gray')
    plt.title('Predicted Mask')
    plt.axis('off')

    plt.show()

# Example usage with a validation image
idx = 0  # Change index as needed
predict_and_display(model, X_val[idx], y_val[idx])


In [ ]:
def visualize_predictions(model, images, masks, num=5):
    plt.figure(figsize=(15, num * 5))
    for i in range(num):
        image = images[i]
        mask = masks[i]
        pred_mask = model.predict(image[np.newaxis, ...])[0]
        pred_mask = (pred_mask > 0.5).astype(np.uint8)

        # Original Image
        plt.subplot(num, 3, 3*i + 1)
        plt.imshow(image.squeeze(), cmap='gray')
        plt.title('Original Image')
        plt.axis('off')

        # Ground Truth Mask
        plt.subplot(num, 3, 3*i + 2)
        plt.imshow(mask.squeeze(), cmap='gray')
        plt.title('Ground Truth Mask')
        plt.axis('off')

        # Predicted Mask
        plt.subplot(num, 3, 3*i + 3)
        plt.imshow(pred_mask.squeeze(), cmap='gray')
        plt.title('Predicted Mask')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize first 5 predictions
visualize_predictions(model, X_val, y_val, num=5)
